# Week 23 · Notebook 1: MLflow, Feature Engineering & XGBoost ETA

# Requirements: Databricks workspace (free trial) + upload the week-01 CSVs to a volume

Upload into `/Volumes/zrl_/zorologistics/raw/`:

- `shipments.csv`
- `carriers.csv`
- `lanes.csv`

Run on a **DBR ML** cluster (single node is fine). This notebook logs an XGBoost ETA model, builds UC feature tables with a point-in-time join, registers the model, and promotes it with `@prod`.


## What this notebook builds

1. **UC feature tables**: `carrier_features` and `lane_features` (static) plus `carrier_daily_stats` (time-series).
2. A **point-in-time training set** via `FeatureEngineeringClient` + `FeatureLookup`, the time-series feature joins *AS OF* each shipment's planned departure, so the model never sees the future.
3. An XGBoost **ETA regression** (predicting `delay_hours`) with MLflow autologging, registered to **Models in Unity Catalog** and promoted with a movable **alias** (`@prod` vs `@challenger`).

See research §6 and module files `07`, `09` in `reference/platforms/databricks/`.


In [ ]:
import pandas as pd
import mlflow
import mlflow.xgboost
from mlflow import MlflowClient
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql import functions as F

fe = FeatureEngineeringClient()
print("mlflow version:", mlflow.__version__)


## Load the raw data

Read the three CSVs from the volume. The **label** is `delay_hours` (ETA deviation); the **lookup keys** are `carrier_id` / `lane_id`; the **timestamp key** for point-in-time is `planned_departure`. We sample 20% for a fast, deterministic run.


In [ ]:
VOL = "/Volumes/zrl_/zorologistics/raw"

ship = (spark.read.format("csv").option("header", True).option("inferSchema", True)
        .load(f"{VOL}/shipments.csv"))
car  = (spark.read.format("csv").option("header", True).option("inferSchema", True)
        .load(f"{VOL}/carriers.csv"))
lan  = (spark.read.format("csv").option("header", True).option("inferSchema", True)
        .load(f"{VOL}/lanes.csv"))

labels = (ship
  .withColumn("planned_departure",
              F.to_timestamp("planned_departure", "yyyy-MM-dd HH:mm:ss[.SSSSSS]"))
  .withColumn("delay_hours", F.col("delay_hours").cast("double"))
  .select("shipment_id", "carrier_id", "lane_id", "planned_departure", "delay_hours")
  .sample(fraction=0.2, seed=42))

print("label rows:", labels.count())


## Build UC feature tables

A **feature table** is a Delta table in UC with a **primary key**. Static features (no time dependency) go straight in; they carry governance and lineage. `fe.create_table` registers them.


In [ ]:
# Static features, create as UC feature tables with primary keys.
fe.create_table(
    name="zrl_.zorologistics.carrier_features",
    primary_keys=["carrier_id"],
    df=car.select("carrier_id", "on_time_rate", "base_rate_usd_per_km_ton", "fleet_size", "region"),
    description="Carrier reliability + rate features",
)

fe.create_table(
    name="zrl_.zorologistics.lane_features",
    primary_keys=["lane_id"],
    df=lan.select("lane_id", "distance_km", "avg_transit_days", "toll_km", "port_region"),
    description="Lane distance/transit features",
)
print("feature tables created")


## A time-series feature table (for point-in-time joins)

`carrier_daily_stats` aggregates per-carrier stats *per day* and declares `timeseries_columns=["metric_date"]`. During training, `timestamp_lookup_key` makes each lookup an **AS OF** join at the label timestamp, computing a carrier's on-time rate using only history *before* the shipment departed. That is the anti-leakage guarantee Week 3 drilled in, now enforced by the platform.


In [ ]:
daily = (ship
  .withColumn("actual_arrival",
              F.to_timestamp("actual_arrival", "yyyy-MM-dd HH:mm:ss[.SSSSSS]"))
  .withColumn("delay_hours", F.col("delay_hours").cast("double"))
  .withColumn("metric_date", F.to_date("actual_arrival"))
  .groupBy("carrier_id", "metric_date")
  .agg(
      F.count("*").alias("daily_shipments"),
      F.round(F.avg(F.when(F.col("delay_hours") <= 2.0, 1.0).otherwise(0.0)), 4).alias("daily_on_time_rate"),
      F.round(F.avg("delay_hours"), 2).alias("daily_avg_delay"),
  ))

fe.create_table(
    name="zrl_.zorologistics.carrier_daily_stats",
    primary_keys=["carrier_id", "metric_date"],
    timeseries_columns=["metric_date"],
    df=daily,
    description="Point-in-time carrier stats (time-series feature table)",
)
print("time-series feature table created")


## Assemble the training set with FeatureLookup

`create_training_set` joins the label DataFrame to the feature tables. Two lookups are keyless on the entity; the third is a **point-in-time** lookup on `carrier_daily_stats` using `planned_departure` as the AS-OF timestamp. We exclude the entity id and the timestamp key from the feature matrix.


In [ ]:
training_set = fe.create_training_set(
    df=labels,
    label="delay_hours",
    feature_lookups=[
        FeatureLookup(table_name="zrl_.zorologistics.carrier_features", lookup_key="carrier_id"),
        FeatureLookup(table_name="zrl_.zorologistics.lane_features", lookup_key="lane_id"),
        FeatureLookup(
            table_name="zrl_.zorologistics.carrier_daily_stats",
            lookup_key="carrier_id",
            timestamp_lookup_key="planned_departure",
        ),
    ],
    exclude_columns=["shipment_id", "planned_departure"],
)

X = training_set.load_df().toPandas()
print("training set shape:", X.shape)
print("feature columns:", sorted(c for c in X.columns if c != "delay_hours"))


## Train the champion with MLflow autologging

`mlflow.xgboost.autolog()` records parameters, metrics, and the model automatically. We split time-agnostically here for brevity; for a real ETA model prefer a time-aware split. The model is registered to **Models in Unity Catalog** under `zrl_.zorologistics.eta_model`.


In [ ]:
y = X["delay_hours"]
feats = X.drop(columns=["delay_hours"])
X_tr, X_te, y_tr, y_te = train_test_split(feats, y, test_size=0.2, random_state=42)

mlflow.set_experiment("/zrl_/eta_experiment")
mlflow.xgboost.autolog()

with mlflow.start_run(run_name="champion") as run:
    model = xgb.XGBRegressor(n_estimators=200, max_depth=6,
                             learning_rate=0.1, random_state=42)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    rmse = mean_squared_error(y_te, pred) ** 0.5
    mlflow.log_metrics({"test_mae": mae, "test_rmse": rmse})
    champion_run_id = run.info.run_id
    champion_version = mlflow.register_model(
        f"runs:/{champion_run_id}/model", "zrl_.zorologistics.eta_model").version

print("champion MAE (hours):", round(mae, 4))
print("champion RMSE (hours):", round(rmse, 4))
print("champion version:", champion_version)


## Train the challenger (static features only)

To *measure the value of the point-in-time feature*, the challenger drops the `daily_*` columns and trains on static features only. Comparing MAE tells you what the time-series lookup is worth.


In [ ]:
daily_cols = [c for c in feats.columns if c.startswith("daily_")]

with mlflow.start_run(run_name="challenger"):
    model2 = xgb.XGBRegressor(n_estimators=200, max_depth=6,
                              learning_rate=0.1, random_state=42)
    model2.fit(X_tr.drop(columns=daily_cols), y_tr)
    pred2 = model2.predict(X_te.drop(columns=daily_cols))
    mae2 = mean_absolute_error(y_te, pred2)
    challenger_run_id = mlflow.active_run().info.run_id
    challenger_version = mlflow.register_model(
        f"runs:/{challenger_run_id}/model", "zrl_.zorologistics.eta_model").version

print("challenger MAE (hours):", round(mae2, 4))
print("challenger version:", challenger_version)


## Promote with aliases: `@prod` vs `@challenger`

Stages are deprecated, use **movable aliases**. The better model gets `@prod`; the other stays `@challenger` for safe comparison.


In [ ]:
client = MlflowClient()
name = "zrl_.zorologistics.eta_model"

best_version  = champion_version  if mae < mae2 else challenger_version
other_version = challenger_version if mae < mae2 else champion_version

client.set_registered_model_alias(name, "@prod", best_version)
client.set_registered_model_alias(name, "@challenger", other_version)

print(f"@prod -> version {best_version}")
print(f"@challenger -> version {other_version}")
print("winner by test MAE:", "champion" if mae < mae2 else "challenger")


In [ ]:
# Final metric: the two test MAEs and the feature count.
print("champion   test MAE (hours):", round(mae, 4))
print("challenger test MAE (hours):", round(mae2, 4))
print("point-in-time gain (hours):", round(mae2 - mae, 4))
print("features used (champion):", len(feats.columns))
